#### Environment Check

In [33]:
import sys
from pathlib import Path

LAB_DIR = Path.cwd()

if LAB_DIR.name == "notebooks":
    LAB_DIR = LAB_DIR.parent

CODE_DIR = LAB_DIR / "code"

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

print("Python:", sys.executable)
print("Lab directory:", LAB_DIR)
print("Code directory:", CODE_DIR)

Python: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python
Lab directory: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/06-best-practices/retrieval-lab
Code directory: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/06-best-practices/retrieval-lab/code


#### Check Elasticsearch

In [34]:
from elasticsearch import ConnectionError, Elasticsearch

ES_URL = "http://localhost:9200"
es_client = Elasticsearch(ES_URL)

try:
    info = es_client.info()
    print("Elasticsearch:", info["version"]["number"])
except ConnectionError as err:
    raise RuntimeError(
        "Elasticsearch is not running. From retrieval-lab, run: docker compose up -d"
    ) from err

Elasticsearch: 8.19.3


#### Check Search Index

In [36]:
INDEX_NAME = "course-questions"

index_exists = es_client.indices.exists(index=INDEX_NAME)
print("Index exists:", index_exists)

if not index_exists:
    raise RuntimeError(
        "The course-questions index is missing. From retrieval-lab, run: uv run python code/ingest.py"
    )

Index exists: True


#### Load LangChain Retriever

In [37]:
from langchain_retriever import create_hybrid_retriever

hybrid_retriever = create_hybrid_retriever()
print("Retriever loaded:", type(hybrid_retriever).__name__)

Retriever loaded: ElasticsearchRetriever


#### Run Hybrid Search With LangChain

In [38]:
query = "I just discovered the course. Can I still join it?"

langchain_results = hybrid_retriever.invoke(query)

len(langchain_results)

5

#### Inspect LangChain Results

In [39]:
for result in langchain_results:
    source = result.metadata["_source"]

    print("Course:", source["course"])
    print("Question:", source["question"])
    print("Score:", result.metadata.get("_score"))
    print("Text:", result.page_content[:300])
    print()

Course: data-engineering-zoomcamp
Question: Course: Can I still join the course after the start date?
Score: 61.132404
Text: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

Course: data-engineering-zoomcamp
Question: Course - Can I follow the course after it finishes?
Score: 43.19504
Text: Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.

You can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.

Course: data-engineering-zoomcamp
Question: Course: What can I do before the course starts?
Score: 32.419098
Text: Get the basic environment ready ahead of time:

- Google Cloud account (free trial — see the GCP setup FAQ).
- Google Cloud SDK (`gcloud` CLI).
- Python 3 — install 

#### Convert LangChain Results To Dictionaries

In [40]:
langchain_documents = []

for result in langchain_results:
    source = result.metadata["_source"]

    langchain_documents.append({
        "id": source["id"],
        "course": source["course"],
        "question": source["question"],
        "text": result.page_content,
        "section": source.get("section", ""),
    })

langchain_documents[0]

{'id': '860',
 'course': 'data-engineering-zoomcamp',
 'question': 'Course: Can I still join the course after the start date?',
 'text': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'section': 'General Course-Related Questions'}

#### Compare With Direct Elasticsearch Hybrid Search

In [41]:
from search import hybrid_search

direct_results = hybrid_search(query, num_results=5)

for doc in direct_results:
    print(doc["course"], "-", doc["question"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

data-engineering-zoomcamp - Course: Can I still join the course after the start date?
data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
data-engineering-zoomcamp - Course: What can I do before the course starts?
data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?
data-engineering-zoomcamp - Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?


#### Compare The Two Interfaces

In [42]:
print("LangChain retriever results:")
for doc in langchain_documents:
    print("-", doc["question"])

print()
print("Direct Elasticsearch hybrid search results:")
for doc in direct_results:
    print("-", doc["question"])

LangChain retriever results:
- Course: Can I still join the course after the start date?
- Course - Can I follow the course after it finishes?
- Course: What can I do before the course starts?
- Course: Can I get support if I take the course in the self-paced mode?
- Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?

Direct Elasticsearch hybrid search results:
- Course: Can I still join the course after the start date?
- Course - Can I follow the course after it finishes?
- Course: What can I do before the course starts?
- Course: Can I get support if I take the course in the self-paced mode?
- Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?


#### Short Learning Summary

In [43]:
summary = """
LangChain does not replace the retrieval logic.

In this notebook, LangChain wraps the same Elasticsearch hybrid search idea:
keyword search + vector search.

The main difference is the interface:
- Direct Elasticsearch returns raw search hits.
- LangChain returns Document objects.
""".strip()

print(summary)

LangChain does not replace the retrieval logic.

In this notebook, LangChain wraps the same Elasticsearch hybrid search idea:
keyword search + vector search.

The main difference is the interface:
- Direct Elasticsearch returns raw search hits.
- LangChain returns Document objects.
